# PyTorch 零基础 5/6：nn.Module 与训练、验证循环

这是第 1～41 课 ASR 主线之前的桥梁课。先预测，再运行；看懂输出后必须改一个值验证自己的解释。

| 项目 | 内容 |
|---|---|
| 前置要求 | 完成基础 4；能解释训练循环的五步 |
| 建议投入 | 60～90 分钟，可分两次完成 |
| 核心概念 | nn.Module 与 forward、train/eval 模式、state_dict 与可复现性 |
| 完成标准 | 能解释代码、独立完成练习、从空白重写本课核心函数 |


## 课前诊断（先不要运行代码）

1. 用自己的话解释：nn.Module 与 forward。
2. 猜测 train/eval 模式 最容易出现哪一种错误。
3. 写下你对 state_dict 与可复现性 的暂时理解；不会可以明确写“不知道”。

这三题不计分，只用于留下学习前证据。


## 1. 用 nn.Module 组织有参数的计算


In [1]:
import io
import torch
from torch import nn


class TinyEncoder(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim),
        )

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        if features.ndim != 3:
            raise ValueError(f"expected [B,T,F], got {tuple(features.shape)}")
        return self.network(features)


torch.manual_seed(7)
encoder = TinyEncoder(input_dim=8, hidden_dim=12)
x = torch.randn(2, 5, 8)
y = encoder(x)
print("input/output:", x.shape, y.shape)
assert y.shape == (2, 5, 12)


input/output: torch.Size([2, 5, 8]) torch.Size([2, 5, 12])


## 2. 参数必须注册在 Module 中


In [2]:
parameter_count = sum(parameter.numel() for parameter in encoder.parameters())
print("parameter count:", parameter_count)
for name, parameter in encoder.named_parameters():
    print(name, tuple(parameter.shape))
assert parameter_count > 0


parameter count: 132
network.0.weight (12, 8)
network.0.bias (12,)
network.2.weight (12,)
network.2.bias (12,)


## 3. train/eval 会改变 Dropout、BatchNorm 等层的行为


In [3]:
dropout_model = nn.Sequential(nn.Linear(8, 8), nn.Dropout(p=0.5))
example = torch.ones(64, 8)

dropout_model.train()
train_a = dropout_model(example)
train_b = dropout_model(example)

dropout_model.eval()
with torch.no_grad():
    eval_a = dropout_model(example)
    eval_b = dropout_model(example)

print("train outputs equal:", torch.equal(train_a, train_b))
print("eval outputs equal:", torch.equal(eval_a, eval_b))
assert not torch.equal(train_a, train_b)
assert torch.equal(eval_a, eval_b)


train outputs equal: False
eval outputs equal: True


## 4. 一个可复用的训练步骤


In [4]:
classifier = nn.Linear(4, 3)
optimizer = torch.optim.Adam(classifier.parameters(), lr=1e-2)
loss_fn = nn.CrossEntropyLoss()


def train_step(inputs: torch.Tensor, targets: torch.Tensor) -> float:
    classifier.train()
    logits = classifier(inputs)
    loss = loss_fn(logits, targets)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return float(loss.detach())


inputs = torch.randn(6, 4)
targets = torch.tensor([0, 1, 2, 0, 1, 2])
loss_value = train_step(inputs, targets)
print("train loss:", loss_value)
assert loss_value > 0


train loss: 1.2499889135360718


## 5. state_dict 保存的是可复现的参数状态


In [5]:
buffer = io.BytesIO()
torch.save(encoder.state_dict(), buffer)
buffer.seek(0)

clone = TinyEncoder(input_dim=8, hidden_dim=12)
clone.load_state_dict(torch.load(buffer, weights_only=True))
clone.eval()
encoder.eval()
with torch.no_grad():
    original_output = encoder(x)
    clone_output = clone(x)

assert torch.equal(original_output, clone_output)
print("state_dict round-trip: exact match")


state_dict round-trip: exact match


## 本课练习（保留作答区）


1. `__init__` 与 `forward` 分别负责什么？
2. 为什么层要赋给 `self.xxx`？
3. 预测 `[B,T,8]` 经过 `Linear(8,12)` 的 shape。
4. 比较 train/eval 下 Dropout 输出。
5. 写函数统计总参数量和可训练参数量。
6. 为 TinyEncoder 加分类头并断言 logits shape。
7. 故意传入 `[B,F]`，验证接口错误消息。
8. 实现不更新参数的 `validation_step`。
9. 保存、加载 state_dict 并比较固定输入输出。
10. 列出实验复现至少需要保存的随机种子、代码、参数和数据版本。


评分：每题 0～2 分。达到 16/20 可以继续；12～15 分次日重做错题；低于 12 分回看代码并从空白复现。


## 离场票与间隔复习

- [ ] 我能闭卷解释：nn.Module 与 forward、train/eval 模式、state_dict 与可复现性。
- [ ] 我能预测核心代码的 shape、dtype 或数值方向。
- [ ] 我能从空白重写至少一个函数，并通过正常、边界、错误输入测试。
- [ ] 我能说出一个“代码能运行但语义错误”的例子。

复习安排：明天闭卷回忆 5 分钟；7 天后重做第 4、7、10 题；30 天后重新构造最小实验。

下一步：基础 6：Dataset、DataLoader、变长语音 padding 与 mask。
